In [17]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

**Dataset utilizado**
- El dataset elegido para este proyecto es 'Formula 1 World Championship (1950 - 2024)' el cual consiste de información de carreras, pilotos, escuderias, qualis, cicuitos, tiempos de vuelta, etc. de los campeonatos desde 1950 hasta 2024.

In [3]:
circuits = pd.read_csv('/content/circuits.csv')
constructors = pd.read_csv('/content/constructors.csv')
drivers = pd.read_csv('/content/drivers.csv')
races = pd.read_csv('/content/races.csv')
results = pd.read_csv('/content/results.csv')

print(f'circuits: {circuits.shape}')
print(f'constructors: {constructors.shape}')
print(f'drivers: {drivers.shape}')
print(f'races: {races.shape}')
print(f'results: {results.shape}')

circuits: (77, 9)
constructors: (212, 5)
drivers: (861, 9)
races: (1125, 18)
results: (26759, 18)


In [4]:
# Selección de columnas de interés de cada set de datos
results_selected = results[
    [
        "raceId",
        "driverId",
        "constructorId",
        "grid",
        "positionOrder",
        "points",
        "laps",
        "statusId"
    ]
]

races_selected = races[
    [
        "raceId",
        "year",
        "round",
        "circuitId"
    ]
]

drivers_selected = drivers[
    [
        "driverId",
        "driverRef",
        "nationality"
    ]
]

constructors_selected = constructors[
    [
        "constructorId",
        "constructorRef"
    ]
]

circuits_selected = circuits[
    [
        "circuitId",
        "circuitRef",
        "country"
    ]
]

In [5]:
# Merge datasets utilizando las llaves primarias
df = results_selected.merge(races_selected, on="raceId", how="left")
df = df.merge(drivers_selected, on="driverId", how="left")
df = df.merge(constructors_selected, on="constructorId", how="left")
df = df.merge(circuits_selected, on="circuitId", how="left")


print("\nMerged dataset shape:", df.shape)
print(df.head())


Merged dataset shape: (26759, 16)
   raceId  driverId  constructorId  grid  positionOrder  points  laps  \
0      18         1              1     1              1    10.0    58   
1      18         2              2     5              2     8.0    58   
2      18         3              3     7              3     6.0    58   
3      18         4              4    11              4     5.0    58   
4      18         5              1     3              5     4.0    58   

   statusId  year  round  circuitId   driverRef nationality constructorRef  \
0         1  2008      1          1    hamilton     British        mclaren   
1         1  2008      1          1    heidfeld      German     bmw_sauber   
2         1  2008      1          1     rosberg      German       williams   
3         1  2008      1          1      alonso     Spanish        renault   
4         1  2008      1          1  kovalainen     Finnish        mclaren   

    circuitRef    country  
0  albert_park  Australia  
1

In [6]:
# check nulls
print(df.isnull().sum())

raceId            0
driverId          0
constructorId     0
grid              0
positionOrder     0
points            0
laps              0
statusId          0
year              0
round             0
circuitId         0
driverRef         0
nationality       0
constructorRef    0
circuitRef        0
country           0
dtype: int64


## Filtro temporal

Se decidió utilizar datos desde el año 2010 hasta 2024 para reducir variaciones históricas en reglas, sistemas de puntuación y cambios técnicos de la Fórmula 1. Esta decisión también se alinea con el paper base, el cual trabaja con datos recientes para mantener mayor consistencia en el análisis.

In [14]:
# discard non-qualified, started from pit, disqualified drivers
df = df[df["grid"] > 0]

# use data since 2010
df = df[df["year"] >= 2010]

print("\nMerged dataset shape:", df.shape)
print(f"Dataset: {len(df):,} registros")
print(f"   Años: {df['year'].min()}-{df['year'].max()}")


Merged dataset shape: (6351, 16)
Dataset: 6,351 registros
   Años: 2010-2024


## Variable objetivo

La variable objetivo del proyecto es `positionOrder`, la cual representa la posición final obtenida por un piloto en una carrera.

El problema se plantea como una tarea de regresión, ya que se busca predecir un valor numérico correspondiente a la posición final.

In [15]:
# elegir columna a predecir y features
target_column = "positionOrder"

features = [
    "year",
    "round",
    "grid",
    "points",
    "laps",
    "statusId",
    "driverRef",
    "nationality",
    "constructorRef",
    "circuitRef",
    "country"
]

X = df[features]
y = df[target_column]

In [16]:
# normalizar columnas categóricas
categorical_columns = [
    "driverRef",
    "nationality",
    "constructorRef",
    "circuitRef",
    "country"
]

X_encoded = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True
)

In [19]:
# definir split train/test 80-20
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTrain/Test split:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)



Train/Test split:
X_train: (5080, 197)
X_test: (1271, 197)
y_train: (5080,)
y_test: (1271,)


In [20]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns
)

In [24]:
train_data = X_train_scaled_df.copy()
train_data[target_column] = y_train.reset_index(drop=True)

test_data = X_test_scaled_df.copy()
test_data[target_column] = y_test.reset_index(drop=True)

print("\nTrain/Test split:")
print("train_data:", train_data.shape)
print("test_data:", test_data.shape)


Train/Test split:
train_data: (5080, 198)
test_data: (1271, 198)
